# Algorithm Benchmark

This notebook compares RSA, ML-KEM, ML-DSA, SLH-DSA, and HQC for the payment transaction security use case.

Implementation note: this notebook does not duplicate benchmark logic. It imports and visualizes functions from `benchmarks.algorithm_benchmark`, so the package module remains the single source of truth.

Notes:
- ML-KEM and HQC are KEM/encryption primitives for protecting payment data keys.
- ML-DSA and SLH-DSA are signature primitives for transaction authorization.
- If `liboqs` or `cryptography` is unavailable, the benchmark module clearly reports `simulation fallback`.

In [ ]:
from pathlib import Path
import importlib
import sys

project_root = Path.cwd()
if (project_root / "src").exists():
    sys.path.insert(0, str(project_root / "src"))
else:
    sys.path.insert(0, str(project_root))

from algorithms.crypto_algorithms import list_algorithm_profiles
import benchmarks.algorithm_benchmark as benchmark_module
benchmark_module = importlib.reload(benchmark_module)

attack_estimates_as_dicts = benchmark_module.attack_estimates_as_dicts
benchmark_results_as_dicts = benchmark_module.benchmark_results_as_dicts
toy_crack_results_as_dicts = benchmark_module.toy_crack_results_as_dicts
CRYPTOGRAPHY_AVAILABLE = benchmark_module.CRYPTOGRAPHY_AVAILABLE
OQS_AVAILABLE = benchmark_module.OQS_AVAILABLE

print(f"cryptography available: {CRYPTOGRAPHY_AVAILABLE}")
print(f"liboqs available: {OQS_AVAILABLE}")

## Security Scorecard

The base score is the starting cryptographic posture score before transaction-specific penalties are applied.

In [ ]:
try:
    import pandas as pd
    profiles_df = pd.DataFrame(list_algorithm_profiles())
    display(profiles_df[[
        "name",
        "primary_use",
        "nist_status",
        "quantum_safe",
        "security_level",
        "base_score",
        "production_role",
    ]])
except ImportError:
    for profile in list_algorithm_profiles():
        print(profile)

## Run Local Benchmark

Increase `iterations` for a more stable comparison. Keep it small when running on an older laptop or without native wheels installed.

In [ ]:
iterations = 5
benchmark_rows = benchmark_results_as_dicts(iterations=iterations)

try:
    benchmark_df = pd.DataFrame(benchmark_rows)
    display(benchmark_df)
except NameError:
    for row in benchmark_rows:
        print(row)

## Toy Crack Demo

This is the actual crack-time demo. It uses the same payment transaction payload for every algorithm label, then brute-forces intentionally tiny demo key spaces. The result is useful for observing relative crack-time growth, but it is **not** a real crack of RSA-2048, ML-KEM, ML-DSA, SLH-DSA, or HQC.

In [ ]:
toy_crack_rows = toy_crack_results_as_dicts()

try:
    toy_crack_df = pd.DataFrame(toy_crack_rows)
    display(toy_crack_df[[
        "algorithm",
        "toy_key_bits",
        "toy_keyspace",
        "crack_time_ms",
        "toy_crack_score",
        "notes",
    ]])
except NameError:
    for row in toy_crack_rows:
        print(row)

## Attack-Time Estimate

This section adds a crack-time-oriented score. It does **not** actually crack RSA-2048 or PQC keys. Instead, it calibrates a local trial-operation rate and estimates attack time from brute-force-equivalent security strength. This makes attack resistance a separate scoring option beside deployment/security posture.

In [ ]:
attack_rows = attack_estimates_as_dicts()

try:
    attack_df = pd.DataFrame(attack_rows)
    display(attack_df[[
        "algorithm",
        "estimated_classical_attack_years",
        "estimated_quantum_attack_years",
        "attack_time_score",
        "attack_model",
    ]])
except NameError:
    for row in attack_rows:
        print(row)

## Payment Security Interpretation

Use **ML-KEM-768** to protect payment payload data keys and **ML-DSA-65** to sign transaction authorization messages. Keep **SLH-DSA** and **HQC** as backup families for crypto-agility.